# Cook Islands 5m DEM creation
Load in LiDAR where it exists and combine with FABDEM to produce 5m DEMs

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pathlib
import rioxarray

import os
import sys
module_path = os.path.abspath(os.path.join("..", "src", "pacific-dems"))
if module_path not in sys.path:
    sys.path.append(module_path)

import create_dem_functions
import islands
import create_data_paths

In [ ]:
import geopandas

In [ ]:
resolution = 5 # in meters
country_name = "cook_islands"

# Create paths and get contry information

In [ ]:
paths = create_data_paths.get_paths(country_name=country_name, resolution=resolution)
islands_dict = islands.get_island(country_name)

In [ ]:
islands_dict.keys()

# Specify any LiDAR data

In [ ]:
# Path of LiDAR DEMs
islands_dict["mangaia"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "Mangaia_topobathy_dem_mosaic_1m_MSL.tif") ##
islands_dict["rarotonga"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "Rarotonga_topobathy_dem_mosaic_1m_MSL.tif") ##
islands_dict["aitutaki_islands"]["lidar"] = [create_dem_functions.load_dem(paths["lidar"] / "aitutaki_topobathy_dem_mosaic_1m_MSL.tif")]
islands_dict["manuae_islands"]["lidar"] = [create_dem_functions.load_dem(paths["lidar"] / "manuae_topobathy_dem_mosaic_1m_MSL.tif")]

islands_dict["atiu"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "atiu_topobathy_dem_mosaic_1m_MSL.tif")
islands_dict["manihiki"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "manhiki_topobathy_dem_mosaic_1m_MSL.tif")

islands_dict["mauke"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "mauke_topobathy_dem_mosaic_1m_MSL.tif")
islands_dict["mitiaro"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "mitiaro_topobathy_dem_mosaic_1m_MSL.tif")
islands_dict["nassau"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "nasau_topobathy_dem_mosaic_1m_MSL.tif")
islands_dict["palmerston"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "palmerston_topobathy_dem_mosaic_1m_MSL.tif")
islands_dict["penrhyn"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "penrhyn_topobathy_dem_mosaic_1m_MSL.tif")
islands_dict["rakahanga"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "rakahanga_topobathy_dem_mosaic_1m_MSL.tif")
islands_dict["pukapuka"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "pukapuka_topobathy_dem_mosaic_1m_MSL.tif")
islands_dict["takutea"]["lidar"] = create_dem_functions.load_dem(paths["lidar"] / "takutea_topobathy_dem_mosaic_1m_MSL.tif")


# Specify if LiDAR only
islands_dict["mangaia"]["lidar_only"] = True; islands_dict["mangaia"]["interpolate"] = True
islands_dict["rarotonga"]["lidar_only"] = True; islands_dict["rarotonga"]["interpolate"] = True
islands_dict["aitutaki_islands"]["lidar_only"] = True; islands_dict["aitutaki_islands"]["interpolate"] = True
islands_dict["manuae_islands"]["lidar_only"] = True; islands_dict["manuae_islands"]["interpolate"] = True
islands_dict["suwarrow"]["interpolate"] = True

islands_dict["atiu"]["lidar_only"] = True; islands_dict["atiu"]["interpolate"] = True
islands_dict["manihiki"]["lidar_only"] = True; islands_dict["manihiki"]["interpolate"] = True

islands_dict["mauke"]["lidar_only"] = True; islands_dict["mauke"]["interpolate"] = True
islands_dict["mitiaro"]["lidar_only"] = True; islands_dict["mitiaro"]["interpolate"] = True
islands_dict["nassau"]["lidar_only"] = True; islands_dict["nassau"]["interpolate"] = True
islands_dict["palmerston"]["lidar_only"] = True; islands_dict["palmerston"]["interpolate"] = True
islands_dict["penrhyn"]["lidar_only"] = True; islands_dict["penrhyn"]["interpolate"] = True
islands_dict["pukapuka"]["lidar_only"] = True; islands_dict["pukapuka"]["interpolate"] = True
islands_dict["rakahanga"]["lidar_only"] = True; islands_dict["rakahanga"]["interpolate"] = True
islands_dict["takutea"]["lidar_only"] = True; islands_dict["takutea"]["interpolate"] = True

In [ ]:
islands_dict.pop("takutea")

# Get island outlines, ignore FABDEMs (except for Suwarrow) and create DEM

In [ ]:
for island_name, island_dict in islands_dict.items():
    print(f"Setup for {island_name}")
    island_dict["name"] = island_name
    island_dict["land"] = create_dem_functions.get_islands_in_boundary(island_dict=island_dict, gadm_path=paths["land"], output_path=paths["output"] / island_name)
    if not "lidar_only" in island_dict or not island_dict["lidar_only"]:
        island_dict["fab"] = create_dem_functions.combine_fabs_in_boundary(island_dict=island_dict, fab_path=paths["fabdem"])
create_dem_functions.creating_dems_all_islands(island_groups=islands_dict, resolution=resolution, output_path=paths["output"])